# Xây dựng cơ sở tri thức RAG

Workflow đầy đủ: audit nguồn → hoàn thiện ngữ cảnh GIR/TT85 và mẫu biểu → canonical corpus → BGE-M3 dense + learned sparse → PostgreSQL/pgvector → integrity/idempotency. Logic dùng chung trong `rag/`; mọi kết quả đọc được ngay trong output notebook này. Cache embeddings/snapshot trong `artifacts/` phục vụ resume, không tạo report rời.

Cần Docker RAG healthy và kernel **Python (AI Agent RAG)**. Cell ingestion ghi vào database RAG cấu hình trong `.env`; xem tên database trước cell apply. Các nguồn pháp lý được kiểm chứng ở mức dữ liệu local, không mặc định đã thẩm định pháp lý bên ngoài.

In [1]:
import os
import sys
from pathlib import Path

candidates = [Path(os.environ["AIAGENT_ROOT"])] if "AIAGENT_ROOT" in os.environ else []
for parent in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([parent, parent / "apps/aiAgent", parent / "frappe-bench/apps/aiAgent"])
ROOT = next((p.resolve() for p in candidates if (p / "rag/data.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Set AIAGENT_ROOT to the absolute aiAgent folder")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from rag.data import config  # noqa: E402 - project root must be resolved first

cfg = config()
print("Workspace:", ROOT)

Workspace: /Users/capkimkhanh/Documents/DUT4_1/CSHTTT_Project/frappe-bench/apps/aiAgent


In [2]:
import time

import numpy as np
import pandas as pd
from IPython.display import display
from IPython.utils.io import capture_output

from rag.data import audit, build_corpus
from rag.db import connect, counts, ingest, migrate
from rag.embedding import Embedder, load_embeddings

## 1. Audit và truy vết nguồn
TT31 ingest được nối với master; GIR combined và TT85 source là allowlist. Data/test được dùng kiểm coverage mã, không đưa vào corpus.

In [3]:
audit_result = audit()
display(pd.DataFrame(audit_result["quality"]).T[["count", "duplicate_wording"]])
display(
    pd.DataFrame(
        [
            {
                "scope_chapters": audit_result["scope_count"],
                "graph_edges": audit_result["graph_edges"],
                "broken_endpoints": len(audit_result["broken_endpoints"]),
                "gold_codes_missing": len(audit_result["gold_missing"]),
                "TT85_SHA256_match": audit_result["tt85_source_hash_matches"],
                "legal_verification": audit_result["legal_verification"],
            }
        ]
    )
)

,count,duplicate_wording
tt31,3421,76
gir,115,0
tt85,100,0


,scope_chapters,graph_edges,broken_endpoints,gold_codes_missing,TT85_SHA256_match,legal_verification
0,97,3762,0,0,True,LOCAL_METADATA_ONLY


## 2. GIR context, TT85 hierarchy và 5 mẫu biểu
Các joins GIR được đối chiếu với wording kề bên: phần dẫn → ví dụ, ví dụ → kết luận. Không ghép xuyên rule. TT85 nối đủ khoản → điểm → subpoint, và thêm nội dung tiêu đề/trường của mẫu biểu từ file source. Nguyên văn atomic được lưu riêng với embedding view. Các template chưa được dựng lại layout biểu mẫu gốc.

In [4]:
bundle = build_corpus()
records = pd.DataFrame(bundle["records"])
display(records.groupby(["dataset", "role"]).size().rename("records").reset_index())
print("Snapshot:", bundle["snapshot"])
print("Corpus count:", len(records))
assert len(records) == 3738
assert records["id"].is_unique

,dataset,role,records
0,gir,evidence,115
1,scope,scope,97
2,tt31,candidate,2764
3,tt31,evidence,657
4,tt85,evidence,105


Snapshot: 60a8f8c6cda5d6af3116c813
Corpus count: 3738


In [5]:
examples = [
    r
    for r in bundle["records"]
    if r["id"] in ["gir:GIR_3_NOTE_038", "gir:GIR_5_NOTE_016", "tt85:TT85-A9-3.a.1"]
]
display(
    pd.DataFrame(
        [
            {
                "id": r["id"],
                "source_text": r["source_text"],
                "embedding_view": r["embedding_text"],
                "locator": r["locator"],
            }
            for r in examples
        ]
    )
)
forms = [r for r in bundle["records"] if r["id"].startswith("tt85:TT85-FORM-")]
display(
    pd.DataFrame(
        [
            {
                "id": r["id"],
                "title": r["metadata"]["title"],
                "form_code": r["locator"]["form_code"],
                "fields": len(r["metadata"]["fields"]),
            }
            for r in forms
        ]
    )
)
assert len(forms) == 5

,id,source_text,embedding_view,locator
0,gir:GIR_3_NOTE_038,Phân loại vào nhóm 16.02.,GIR Quy tắc 3 | GIR_EXPLANATORY | (a) Bộ thực...,"{'document': 'Phụ lục II - 6_GIR.pdf', 'pages'..."
1,gir:GIR_5_NOTE_016,(Bao bì),GIR Quy tắc 5 | GIR_EXPLANATORY | (Bao bì)\nQ...,"{'document': 'Phụ lục II - 6_GIR.pdf', 'pages'..."
2,tt85:TT85-A9-3.a.1,a.1) Cơ quan hải quan thực hiện tiếp nhận và k...,"85/2026/TT-BTC | Điều 9: Hồ sơ, thủ tục đối vớ...","{'document': '85/2026/TT-BTC', 'article': 9, '..."


,id,title,form_code,fields
0,tt85:TT85-FORM-01,"Phiếu yêu cầu phân tích hàng hóa xuất khẩu, nh...",01/PYCPT/2026,14
1,tt85:TT85-FORM-02,Phiếu tiếp nhận yêu cầu phân tích hàng hóa xuấ...,02/PTNYCPT/2026,9
2,tt85:TT85-FORM-03,Biên bản trả lại mẫu đã tiếp nhận yêu cầu phân...,03/BBTLMHH/2026,11
3,tt85:TT85-FORM-04,"Thông báo về kết quả phân tích, phân loại đối ...",04/TBKQPTPL/2026,15
4,tt85:TT85-FORM-05,"Phiếu ghi số, ngày văn bản, chứng từ thuộc hồ ...",05/PGTL/2026,6


## 3. BGE-M3 dense + sparse
Pin model/tokenizer commit. Corpus input không được truncate im lặng. Checkpoint theo batch/input hash giúp tái chạy nhanh. CPU FP32 và sparse là cấu hình đã chọn cho experiment; dense là 1.024 chiều, sparse là token weights của model.

In [6]:
embedder = Embedder(cfg)
started = time.perf_counter()
with capture_output() as embedding_progress:
    embedding_manifest = embedder.corpus(bundle)
manifest, vectors, sparse = load_embeddings(bundle)
display(
    pd.DataFrame(
        [
            {
                "records": len(vectors),
                "dimension": vectors.shape[1],
                "max_input_tokens": manifest["max_tokens"],
                "sparse_nonzero": sum(len(w) for w in sparse),
                "elapsed_seconds": round(time.perf_counter() - started, 2),
                "model_revision": manifest["profile"]["revision"],
                "profile": manifest["profile_id"],
            }
        ]
    )
)
np.testing.assert_allclose(np.linalg.norm(vectors, axis=1), 1, atol=1e-5)
assert len(sparse) == len(vectors)
assert np.isfinite(vectors).all()

/Users/capkimkhanh/Documents/DUT4_1/CSHTTT_Project/frappe-bench/apps/aiAgent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching 22 files:   0%|          | 0/22 [00:00<?, ?it/s]

Fetching 22 files: 100%|██████████| 22/22 [00:00<00:00, 4029.99it/s]

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


,records,dimension,max_input_tokens,sparse_nonzero,elapsed_seconds,model_revision,profile
0,3738,1024,376,181202,141.83,5617a9f61b028005a4858fdac845db406aefb181,3a2ce6e6d726a0f053748717


## 4. PostgreSQL ingestion và snapshot activation
Chỉ snapshot đã đủ nguồn, vectors và graph mới được chuyển active. Snapshot cũ được giữ để rollback. `.env` không xuất hiện trong output.

In [7]:
conn = connect()
migrate(conn)
display(
    pd.DataFrame(
        [conn.execute("SELECT current_database() AS database, version() AS server").fetchone()]
    )
)
APPLY = True
print("Apply to the displayed RAG database:", APPLY)

,database,server
0,hs_rag,PostgreSQL 17.6 (Debian 17.6-1.pgdg12+1) on aa...


Apply to the displayed RAG database: True


In [8]:
ingestion_result = ingest(conn, bundle, manifest, vectors, sparse, apply=APPLY)
display(pd.DataFrame([ingestion_result.get("counts", ingestion_result.get("expected"))]))
if APPLY:
    before = counts(conn, bundle["snapshot"], manifest["profile_id"])
    ingest(conn, bundle, manifest, vectors, sparse, apply=True)
    after = counts(conn, bundle["snapshot"], manifest["profile_id"])
    assert before == after
    print("Idempotency: PASS")
    display(pd.DataFrame([conn.execute("SELECT * FROM rag.active").fetchone()]))

,chunks,embeddings,nodes,edges
0,3738,3738,3424,3762


Idempotency: PASS


,singleton,snapshot,profile
0,True,60a8f8c6cda5d6af3116c813,3a2ce6e6d726a0f053748717


## 5. Kiểm chứng cuối
Counts, FK integrity, vector norms và sparse coverage đều được kiểm trực tiếp. Notebook 02 dùng snapshot/profile active này; không cần giữ kernel hiện tại.

In [9]:
if APPLY:
    check = conn.execute(
        """SELECT count(*) AS sparse_postings FROM rag.sparse_postings
        WHERE snapshot=%s AND profile=%s""",
        (bundle["snapshot"], manifest["profile_id"]),
    ).fetchone()
    assert check["sparse_postings"] == sum(len(w) for w in sparse)
    print(
        "Knowledge base ready:", {"snapshot": bundle["snapshot"], "records": len(vectors), **check}
    )
conn.close()

Knowledge base ready: {'snapshot': '60a8f8c6cda5d6af3116c813', 'records': 3738, 'sparse_postings': 181202}
